## Initialisation

Ce notebook construit la couche **silver** à partir des données bronze validées (`valid_parsing = True`).

On commence par établir la connexion à la base PostgreSQL, puis on charge les objets bronze depuis les 4 tables sources.

In [26]:
from sqlalchemy import create_engine, text, URL

DB_USER = "indusense_user"
DB_PASSWORD = "ThEP@ssW0rd"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "indusense_db"

url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

engine = create_engine(url)

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print(f"✅ Connexion à PostgreSQL réussie ({DB_HOST}:{DB_PORT}/{DB_NAME})")
except Exception as e:
    print(f"❌ Échec de connexion : {e}")

✅ Connexion à PostgreSQL réussie (localhost:5432/indusense_db)


In [27]:
from sqlalchemy.orm import Session
from sqlalchemy import select
from models.bronze import BronzeRelevesIncidents, BronzeTelemetry, BronzeMachine, BronzeMaintenance

with Session(engine) as session:
    incidents_raw = session.execute(select(BronzeRelevesIncidents)).scalars().all()
    telemetry_raw = session.execute(select(BronzeTelemetry)).scalars().all()
    machines_raw = session.execute(select(BronzeMachine)).scalars().all()
    maintenances_raw = session.execute(select(BronzeMaintenance)).scalars().all()

    valid_bronze_incidents = session.execute(
        select(BronzeRelevesIncidents).where(BronzeRelevesIncidents.valid_parsing.is_(True))
    ).scalars().all()

    valid_bronze_telemetry = session.execute(
        select(BronzeTelemetry).where(BronzeTelemetry.valid_parsing.is_(True))
    ).scalars().all()

    valid_bronze_machines = machines_raw
    valid_bronze_maintenances = maintenances_raw

    session.expunge_all()

print(f"✅ bronze_releves_incidents : {len(incidents_raw)} lignes chargées")
print(f"✅ bronze_telemetry         : {len(telemetry_raw)} lignes chargées")
print(f"✅ bronze_machine           : {len(valid_bronze_machines)} lignes chargées")
print(f"✅ bronze_maintenance       : {len(valid_bronze_maintenances)} lignes chargées")

✅ bronze_releves_incidents : 1245 lignes chargées
✅ bronze_telemetry         : 135626 lignes chargées
✅ bronze_machine           : 15 lignes chargées
✅ bronze_maintenance       : 1562 lignes chargées


## Nettoyage bronze → silver

Filtrage des lignes invalides (`valid_parsing = False`) avant insertion dans les tables silver.

In [28]:
incidents_filtered = len(incidents_raw) - len(valid_bronze_incidents)
telemetry_filtered = len(telemetry_raw) - len(valid_bronze_telemetry)

print(f"bronze_releves_incidents : {len(incidents_raw)} → {len(valid_bronze_incidents)} lignes valides ({incidents_filtered} filtrées)")
print(f"bronze_telemetry         : {len(telemetry_raw)} → {len(valid_bronze_telemetry)} lignes valides ({telemetry_filtered} filtrées)")
print(f"bronze_machine           : {len(valid_bronze_machines)} lignes (données de référence, pas de filtrage)")
print(f"bronze_maintenance       : {len(valid_bronze_maintenances)} lignes (données de référence, pas de filtrage)")

bronze_releves_incidents : 1245 → 1245 lignes valides (0 filtrées)
bronze_telemetry         : 135626 → 134280 lignes valides (1346 filtrées)
bronze_machine           : 15 lignes (données de référence, pas de filtrage)
bronze_maintenance       : 1562 lignes (données de référence, pas de filtrage)


## Insertion dans les tables silver

Création des tables silver si elles n'existent pas, nettoyage des données existantes, puis insertion des lignes filtrées.

In [29]:
from sqlalchemy import delete
from models.silver import SilverRelevesIncidents, SilverTelemetry, SilverMachine, SilverMaintenance

with Session(engine) as session:
    session.execute(delete(SilverRelevesIncidents))
    session.execute(delete(SilverTelemetry))
    session.execute(delete(SilverMachine))
    session.execute(delete(SilverMaintenance))

    session.add_all([SilverRelevesIncidents(
        incident_id=row.incident_id,
        date=row.date,
        operator_name=row.operator_name,
        machine_id=row.machine_id,
        severity=row.severity,
        operator_badge=row.operator_badge,
        comment=row.comment,
        shift=row.shift,
        type_surchauffe=row.type_surchauffe,
        type_baisse_pression=row.type_baisse_pression,
        type_vibration=row.type_vibration,
        type_bruit_mecanique=row.type_bruit_mecanique,
        type_surconsommation=row.type_surconsommation,
        type_blocage_mecanique=row.type_blocage_mecanique,
        type_alarme_capteur=row.type_alarme_capteur,
        type_arret_urgence=row.type_arret_urgence,
        type_defaut_qualite=row.type_defaut_qualite,
    ) for row in valid_bronze_incidents])

    session.add_all([SilverTelemetry(
        machine_id=row.machine_id,
        date=row.date,
        temperature_c=row.temperature_c,
        pressure_bar=row.pressure_bar,
        voltage_mean_v=row.voltage_mean_v,
        rotation_mean_rpm=row.rotation_mean_rpm,
        pieces_produced=row.pieces_produced,
    ) for row in valid_bronze_telemetry])

    session.add_all([SilverMachine(
        machine_code=row.machine_code,
        commissioning_date=row.commissioning_date,
        max_daily_capacity=row.max_daily_capacity,
        max_hourly_capacity_pieces=row.max_hourly_capacity_pieces,
        model=row.model,
        production_line=row.production_line,
        location=row.location,
        criticality=row.criticality,
        is_active=row.is_active,
        created_at=row.created_at,
        updated_at=row.updated_at,
    ) for row in valid_bronze_machines])

    session.add_all([SilverMaintenance(
        maintenance_id=row.maintenance_id,
        machine_code=row.machine_code,
        maintenance_at=row.maintenance_at,
        maintenance_type=row.maintenance_type,
        action_type=row.action_type,
        component=row.component,
        description=row.description,
        related_incident_id=row.related_incident_id,
        duration_hours=row.duration_hours,
        created_at=row.created_at,
        updated_at=row.updated_at,
    ) for row in valid_bronze_maintenances])

    session.commit()

print(f"✅ silver_releves_incidents : {len(valid_bronze_incidents)} lignes insérées")
print(f"✅ silver_telemetry         : {len(valid_bronze_telemetry)} lignes insérées")
print(f"✅ silver_machine           : {len(valid_bronze_machines)} lignes insérées")
print(f"✅ silver_maintenance       : {len(valid_bronze_maintenances)} lignes insérées")

✅ silver_releves_incidents : 1245 lignes insérées
✅ silver_telemetry         : 134280 lignes insérées
✅ silver_machine           : 15 lignes insérées
✅ silver_maintenance       : 1562 lignes insérées
